In [1]:
# importing useful packages and functions
import pandas as pd
import numpy as np
from astropy.cosmology import FlatLambdaCDM
import bilby
from scipy.integrate import cumtrapz
from scipy.interpolate import interp1d
from astropy import constants as const
from scipy.stats import ncx2
import copy
import h5py

In [34]:
#set the number of injections
nsample = 1000

In [35]:
# sample parameters from the binary black hole prior
injection = bilby.gw.prior.BBHPriorDict().sample(nsample)

22:37 bilby INFO    : No prior given, using default BBH priors in /media/SSD/anaconda_dir/anaconda3/lib/python3.11/site-packages/bilby/gw/prior_files/precessing_spins_bbh.prior.


In [36]:
# convert mass ratio and chirp mass to total mass
total_masses = bilby.gw.conversion.chirp_mass_and_mass_ratio_to_total_mass( 
    injection['chirp_mass'], 
    injection['mass_ratio'] )

In [5]:
f = h5py.File('mock_for_sm_abs.hdf5', 'r') 

data = pd.DataFrame( data = [] )
for a in f.keys():
    if a == 'ra':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'dec':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'z':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'z_obs':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'm_K':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'm_B':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'm_W1':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'm_W2':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'stellar_mass':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'sigmaz':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'skymap_indices':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])

dec
m_B
m_K
m_W1
m_W2
ra
sigmaz
skymap_indices
stellar_mass
z
z_obs


In [21]:
data.head()

,dec,m_B,m_K,m_W1,m_W2,ra,sigmaz,skymap_indices,stellar_mass,z,z_obs,dL
0,0.760462,23.418209,20.178024,20.059195,20.059195,0.020387,0.0,10628660,0.053306,0.185029,0.182923,867.781789
1,0.763832,19.848720,17.183575,17.062308,17.062308,0.006789,0.0,10640496,0.618883,0.186787,0.185017,876.997244
2,0.760068,20.054054,16.939637,16.819193,16.819193,0.005451,0.0,10624695,0.843367,0.190252,0.189618,895.221576
3,0.761422,18.432881,14.663945,14.547490,14.547490,0.010789,0.0,10632606,8.170673,0.185415,0.183351,869.802505
4,0.761146,19.344737,16.740426,16.616184,16.616184,0.011293,0.0,10628655,0.811651,0.185504,0.182808,870.268167


In [37]:
# we will need to have luminosity distance
cosmo = FlatLambdaCDM(H0=73, Om0=0.25, Ob0=0.045)
dL = cosmo.luminosity_distance(data.z)
data['dL'] = dL
data.head()

,dec,m_B,m_K,m_W1,m_W2,ra,sigmaz,skymap_indices,stellar_mass,z,z_obs,dL
0,0.760462,23.418209,20.178024,20.059195,20.059195,0.020387,0.0,10628660,0.053306,0.185029,0.182923,867.781789
1,0.763832,19.848720,17.183575,17.062308,17.062308,0.006789,0.0,10640496,0.618883,0.186787,0.185017,876.997244
2,0.760068,20.054054,16.939637,16.819193,16.819193,0.005451,0.0,10624695,0.843367,0.190252,0.189618,895.221576
3,0.761422,18.432881,14.663945,14.547490,14.547490,0.010789,0.0,10632606,8.170673,0.185415,0.183351,869.802505
4,0.761146,19.344737,16.740426,16.616184,16.616184,0.011293,0.0,10628655,0.811651,0.185504,0.182808,870.268167


In [38]:
# weighting with the merger rate evolution
def merger_rate_evolution( z, gamma, k, zp ):
    const = 1+(1+zp)**(-gamma-k)
    frac = (1+z)/(1+zp)
    
    return (const*(1+z)**gamma) / (1+frac**(gamma+k))

R_z = merger_rate_evolution(data.z, 4.56, 2.86, 2.47 )/(1+data.z)

In [39]:
# weighting with stellar mass
weights = R_z*data.stellar_mass
weights /= np.sum(weights) # the sum of the weights must be 1
np.sum(weights)

0.9999999999999987

In [40]:
# select the host galaxies based on their stellar mass
selecti = np.random.choice( data.index, size = nsample, p = weights )
data.iloc[selecti]

,dec,m_B,m_K,m_W1,m_W2,ra,sigmaz,skymap_indices,stellar_mass,z,z_obs,dL
61778356,-1.172018,17.656073,13.874308,13.760161,13.760161,1.408695,0.0,493469,8.228153,0.126603,0.126640,571.381670
21889368,0.962844,18.062233,14.358787,14.241343,14.241343,4.656550,0.0,11455634,8.358157,0.162957,0.162456,753.508790
33554617,-0.414317,19.930411,16.870492,16.753865,16.753865,4.758982,0.0,3761182,0.286639,0.097280,0.098258,430.160837
50518084,1.122058,19.958690,16.211959,16.104192,16.104192,2.310226,0.0,11959888,2.106158,0.163825,0.162850,757.953187
56200793,-0.888953,18.089595,14.624818,14.507501,14.507501,5.503501,0.0,1409103,4.861356,0.148248,0.148966,678.897863
...,...,...,...,...,...,...,...,...,...,...,...,...
15701554,-0.769876,23.773434,21.327795,21.216140,21.216140,6.242583,0.0,1914898,0.024110,0.198445,0.199034,938.565836
50974890,0.063585,23.376834,20.143044,20.035667,20.035667,3.446034,0.0,6693062,0.054326,0.156492,0.156220,720.561738
20464776,-0.606605,18.730551,15.034181,14.917991,14.917991,1.735861,0.0,2702443,5.870311,0.181975,0.180307,851.804281
26498380,-0.533060,21.361095,17.794321,17.688870,17.688870,0.407793,0.0,3090698,0.736169,0.154691,0.153332,711.428100


In [41]:
#### Add the parameters of host galaxies
injection['dec'] = np.array( data.iloc[selecti]['dec'] )
injection['ra'] = np.array( data.iloc[selecti]['ra'] )
injection['luminosity_distance'] = np.array( data.iloc[selecti]['dL'] )
injection['galnum'] = data.iloc[selecti].index

Function from https://github.com/simone-mastrogiovanni/cosmic_dipole_GW_3G/blob/main/gwdip.py

In [42]:
def fit_PSD_factor(list_of_ASD,fmin,fmax):
    '''
    Provided a list of txt files with detectors ASDs, it will return the interpolant of the network sensitivity
    for the 0 PN expansion model of the signal
    
    Parameters
    ----------
    list_of_ASD: list
        List of string containing the file names, files should have frequency on first column and ASD in Hz^-0.5 on second
    fmin, fmax: float
        minimum and maximum for the interpolation, outside the range the sensitivity will be zero.
    '''
    
    interpolant=[]
    for ff in list_of_ASD:
        data=np.genfromtxt(ff)
        cum_PSD=cumtrapz(np.power(data[:,0],-7./3)/(data[:,1]**2.),data[:,0])
        interpolant.append(interp1d(data[:-1,0],cum_PSD,bounds_error=False,fill_value=(0.,cum_PSD[-1])))    
    freq_eval=np.logspace(np.log10(fmin),np.log10(fmax),10000)
    evals=np.zeros_like(freq_eval)
    for i in range(len(interpolant)):
        evals+=interpolant[i](freq_eval)
    return interp1d(freq_eval,evals,bounds_error=False,fill_value=(0.,evals[-1]))

In [43]:
Q=0.2*(np.power(0.5*(1+np.cos(injection['theta_jn'])**2.),2.)+np.cos(injection['theta_jn'])**2)
lso = 1./(total_masses*4.9e-6*np.pi*6**1.5)

O4_list = ['aligo_O4high.txt', 'avirgo_O4high_NEW.txt']
interp=fit_PSD_factor(O4_list,10,2048)

In [ ]:
#SNR
num_detectors = 3 #Number of detectors in the network
snr_thr = 11  #SNR threshold for the detection

injection['snr']=( np.power(const.G.value*injection['chirp_mass']*1.99e30,5./3)*Q)/(6*np.power(const.c.value,3.)*np.power(np.pi,4/3.)*np.power(injection['luminosity_distance']*3.08e22,2.))
injection['snr']*=interp(lso)
injection['snr']=np.sqrt(ncx2.rvs(num_detectors,injection['snr']))

injections = pd.DataFrame( injection )
injections = copy.deepcopy( injections[ injections.snr > snr_thr ] )
len(injections)

In [45]:
#injections['galnum'] = injections.index
length = len(injections)
idx = np.linspace( 0, length-1, length )
injections.index = idx.astype('int') 
injections.head()

,mass_ratio,chirp_mass,luminosity_distance,dec,ra,theta_jn,psi,phase,a_1,a_2,tilt_1,tilt_2,phi_12,phi_jl,galnum,snr
0,0.918158,57.784060,518.558691,-1.188815,0.827861,1.006129,1.081351,4.024085,0.733983,0.901443,0.856091,2.461736,2.977000,0.164417,62502950,11.338296
1,0.345727,29.568547,513.563319,-0.102171,0.275706,2.869726,1.058212,4.382543,0.569579,0.828517,1.198935,1.080389,5.784907,5.686742,72216055,17.379351
2,0.360176,47.474141,565.063848,-1.205285,3.830024,0.290618,1.099100,6.177828,0.429680,0.819994,2.816830,0.938635,6.164386,4.489586,57365637,13.672486
3,0.411186,28.884411,446.170730,1.075143,4.121505,2.148172,0.036051,0.964410,0.601633,0.232711,0.648186,1.867833,1.376532,2.841610,13937401,13.272446
4,0.830736,34.160869,710.835963,0.392817,3.418468,0.517427,0.901150,4.475619,0.668891,0.298087,1.949845,0.721789,0.612619,1.551977,26652635,13.290933


In [46]:
#### sample the time
geocent_time = bilby.core.prior.analytical.Uniform( minimum = 1238170292, 
                                                    maximum = 1269360692 ).sample(len(injections))

In [47]:
# convert mass ratio and chirp mass to component masses
total_masses = bilby.gw.conversion.chirp_mass_and_mass_ratio_to_total_mass( 
    injections['chirp_mass'], 
    injections['mass_ratio'] )

mass_1, mass_2 = bilby.gw.conversion.total_mass_and_mass_ratio_to_component_masses(
    injections['mass_ratio'],
    total_masses )

( mass_1 > mass_2 ).sum()

195

In [49]:
# saving injection parameters
for i in range(len(injections)):
    with open("injections_SNR11_sm/injection_params"+str(i)+".csv", "w") as file:
        file.write('mass_1, ' + str(mass_1[i]) + '\n')
        file.write('mass_2, ' + str(mass_2[i]) + '\n')
        file.write('a_1, ' + str(injections['a_1'][i]) + '\n' )
        file.write('a_2, ' + str(injections['a_2'][i]) + '\n' )
        file.write('tilt_1, ' + str(injections['tilt_1'][i]) + '\n' )
        file.write('tilt_2, ' + str(injections['tilt_2'][i]) + '\n' )
        file.write('phi_12, ' + str(injections['phi_12'][i]) + '\n' )
        file.write('phi_jl, ' + str(injections['phi_jl'][i]) + '\n' )
        file.write('luminosity_distance, ' + str(injections['luminosity_distance'][i]) + '\n' )
        file.write('theta_jn, ' + str(injections['theta_jn'][i]) + '\n' )
        file.write('psi, ' + str(injections['psi'][i]) + '\n' )
        file.write('phase, ' + str(injections['phase'][i]) + '\n')
        file.write('geocent_time, ' + str(geocent_time[i]) + '\n')
        file.write('ra, ' + str(injections['ra'][i]) + '\n')
        file.write('dec, ' + str(injections['dec'][i]))
        file.close()

In [50]:
# saving hosts
data.iloc[injections.galnum].to_csv( 'host_for_sm_small.txt', index = False )